# Pipeline de Produção — Selagem FB14

Notebook orquestrador agendável como Seeq job.  
Executa a cada 1 hora: extrai dados, atualiza datasets analíticos, avalia gatilhos e grava no SharePoint.

**Fluxo**
1. Generators: `sku_dates` (Seeq) + `00_hour_prev` → `01_vida_rul` → `02_sinais_forca` → `03_padroes`
2. SKU normalization: adiciona `Media_norm` ao sinal de força
3. Avaliação de gatilhos v3.0: RISCO / CRÍTICO / EMERGENCIAL / RED / AMARELO / REVISAO
4. Gravação SharePoint (`ESCREVER_SP = True`)

In [1]:
import sys
from pathlib import Path

# notebooks/ para mock seeq em dev local; lplcc_selagem/ para imports src.*
_NB_DIR = Path.cwd()
_ROOT   = _NB_DIR.parent
for _p in (_NB_DIR, _ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from seeq import spy
spy.jobs.schedule("every 1 hour")

,Schedule,Scheduled,Next Run
0,every 1 hour,Every hour,2026-05-25 20:00:00 BRT


,Schedule,Scheduled,Next Run
0,every 1 hour,Every hour,2026-05-25 20:00:00 BRT


In [2]:
# ── Configuração ──────────────────────────────────────────────────────────────
MAQUINA     = "FB14"
LIST_NAME   = "Gatilhos_Selagem"
SP_URL      = ""    # defina SP_URL=https://... em sharepoint.ev
ESCREVER_SP = True  # alterar para True em produção

PATH_HOUR_PREV = _NB_DIR / "00_hour_prev.csv"
PATH_VIDA_RUL  = _NB_DIR / "01_vida_rul.csv"
PATH_WEIBULL   = _NB_DIR / "01_weibull_params.json"
PATH_SINAIS    = _NB_DIR / "02_sinais_forca.csv"
PATH_PADROES   = _NB_DIR / "03_padroes.csv"
PATH_SKU_DATES = _NB_DIR / "sku_dates.csv"
PATH_TROCA     = _NB_DIR / "troca_modulo.csv"
PATH_STATE     = _NB_DIR / "state_fb14.json"

## Etapa 1 — Generators

Atualiza os quatro datasets analíticos a partir dos dados mais recentes do Seeq.

In [3]:
from src.generators.gen_sku_dates  import run as gen_sku_dates
from src.generators.gen_hour_prev  import run as gen_hour_prev
from src.generators.gen_vida_rul   import run as gen_vida_rul
from src.generators.gen_sinais     import run as gen_sinais
from src.generators.gen_padroes    import run as gen_padroes

print("[0/4] Atualizando SKU dates via Seeq...")
df_sku_raw = gen_sku_dates(output_path=PATH_SKU_DATES)
n_sku = df_sku_raw["bag1_sku"].notna().sum()
print(f"      sku_dates: {len(df_sku_raw):,} leituras | {n_sku:,} com SKU bag1")

print("[1/4] Extraindo dados do Seeq...")
df_hour = gen_hour_prev(output_path=PATH_HOUR_PREV, troca_csv=PATH_TROCA)
print(f"      00_hour_prev: {len(df_hour):,} leituras")

print("[2/4] Calculando vida e Weibull...")
df_vida = gen_vida_rul(
    input_path=PATH_HOUR_PREV,
    output_csv=PATH_VIDA_RUL,
    output_json=PATH_WEIBULL,
    troca_csv=PATH_TROCA,
)
n_ciclos = df_vida["ciclo_id"].nunique() if "ciclo_id" in df_vida.columns else "?"
print(f"      01_vida_rul: {len(df_vida):,} linhas | {n_ciclos} ciclos")

print("[3/4] Calculando features de força...")
df_sinais = gen_sinais(
    input_path=PATH_HOUR_PREV,
    output_path=PATH_SINAIS,
    troca_csv=PATH_TROCA,
)
print(f"      02_sinais_forca: {len(df_sinais):,} linhas")

print("[4/4] Calculando padrões de detecção...")
df_padroes = gen_padroes(
    input_vida_rul=PATH_VIDA_RUL,
    input_sinais=PATH_SINAIS,
    output_path=PATH_PADROES,
    troca_csv=PATH_TROCA,
)
n_gen = df_padroes["genuino"].sum() if "genuino" in df_padroes.columns else "?"
print(f"      03_padroes: {len(df_padroes)} ciclos ({n_gen} genuínos)")

,ID,Type,Time,Count,Pages,Data Processed,Result
0,951157FA-D4BB-4696-A534-AEE4B48532CB,Signal,00:00:01.22,3627,1,58 KB,Success
1,8D9E2FE1-6000-438C-B293-0EDDAA182851,Signal,00:00:01.08,3513,1,56 KB,Success


      00_hour_prev: 3,255 leituras
[2/4] Calculando vida e Weibull...
      01_vida_rul: 3,255 linhas | 16 ciclos
[3/4] Calculando features de força...
      02_sinais_forca: 3,255 linhas
[4/4] Calculando padrões de detecção...
      03_padroes: 14 ciclos (13 genuínos)


## Etapa 2 — Normalização por SKU

Adiciona `Media_norm` ao sinal de força.  
Elimina o Paradoxo de Simpson: SKUs pesados elevam a força bruta mascarando degradação.

In [4]:
import pandas as pd
from src.sku_normalizer import normalizar_media

# Carrega força bruta com UTC
df_raw = pd.read_csv(PATH_HOUR_PREV, parse_dates=["Timestamp"])
df_raw["Timestamp"] = pd.to_datetime(df_raw["Timestamp"], utc=True)
df_raw = df_raw.set_index("Timestamp").sort_index()

# SKU dates → converte para UTC para compatibilidade com merge_asof
df_sku = pd.read_csv(PATH_SKU_DATES, parse_dates=["index"])
df_sku["ts"] = pd.to_datetime(df_sku["index"], utc=True)

df_hourly = normalizar_media(
    df_forca=df_raw,
    df_sku=df_sku,
    col_sku="bag1_sku",
    col_media="Media",
)

n_norm  = df_hourly["Media_norm"].notna().sum()
n_total = len(df_hourly)
cobertura = n_norm / n_total if n_total > 0 else 0.0
print(f"Media_norm: {n_norm:,}/{n_total:,} leituras ({cobertura:.0%})")
if n_norm > 0:
    fator_medio = df_hourly["sku_fator"].mean()
    moda_sku    = df_hourly["sku_codigo"].mode()
    sku_dom     = moda_sku.iloc[0] if not moda_sku.empty else None
    print(f"Fator SKU médio : {fator_medio:.3f}")
    if sku_dom is not None:
        print(f"SKU dominante   : {int(float(sku_dom))}")

# Usa Media_norm se cobertura >= 50%; caso contrário, fallback para força bruta
media_col = "Media_norm" if cobertura >= 0.50 else "Media"
print(f"Coluna p/ trigger: '{media_col}'")

Media_norm: 3,255/3,255 leituras (100%)
Fator SKU médio : 0.983
SKU dominante   : 30244719
Coluna p/ trigger: 'Media_norm'


## Etapa 3 — Estado do ciclo atual

In [5]:
from src.predictor import load_troca_dates

troca_dates  = load_troca_dates(PATH_TROCA)
ultima_troca = troca_dates[-1]
hoje         = df_hourly.index[-1].normalize()
idade_dias   = (hoje - ultima_troca).days

print(f"Última troca confirmada : {ultima_troca.date()}")
print(f"Última leitura          : {hoje.date()}")
print(f"Idade do rolo           : {idade_dias} dias")

Última troca confirmada : 2026-05-19
Última leitura          : 2026-05-25
Idade do rolo           : 6 dias


## Etapa 4 — Avaliação de gatilhos

In [6]:
import os

sp_client = None
if ESCREVER_SP:
    try:
        from dotenv import dotenv_values
        # sharepoint.ev fica na raiz do projeto (fb14new/), não em notebooks/
        _ev = dotenv_values(_ROOT / "sharepoint.ev")
        _sp_url  = _ev.get("SP_URL") or "https://kimberlyclark.sharepoint.com/Sites/H945"
        _sp_user = _ev.get("SP_USER")
        _sp_pass = _ev.get("SP_PASS")
        from src.sharepoint_methods import SharePointClient
        sp_client = SharePointClient(url=_sp_url, username=_sp_user, password=_sp_pass)
        print(f"SharePoint: conectado ({_sp_url})")
    except Exception as _e:
        print(f"SharePoint: falha na conexão ({_e}) — modo dry-run")
else:
    print("SharePoint: dry-run (ESCREVER_SP=False)")

SharePoint: conectado (https://kimberlyclark.sharepoint.com/Sites/H945)


In [7]:
from src.trigger_engine import TriggerEngine

engine = TriggerEngine(maquina=MAQUINA, state_path=PATH_STATE)

print(f"Avaliando {hoje.date()} | rolo com {idade_dias} dias | sinal='{media_col}'")
events = engine.evaluate(
    df_hourly=df_hourly,
    troca_date=ultima_troca,
    sp_client=sp_client,
    list_name=LIST_NAME,
    media_col=media_col,
    troca_dates=troca_dates,
)

print()
if events:
    print(f"{len(events)} disparo(s):")
    for ev in events:
        sp_tag = f" [SP ID={ev.sp_item_id}]" if ev.sp_item_id else ""
        print(f"  [{ev.gatilho}] {ev.severidade}{sp_tag}")
        print(f"  {ev.mensagem.splitlines()[0]}")
else:
    print("Nenhum gatilho disparado.")

Avaliando 2026-05-25 | rolo com 6 dias | sinal='Media_norm'

Nenhum gatilho disparado.


## Diagnóstico (execução manual)

Snapshot completo dos indicadores calculados para hoje. Útil para debug e auditoria.

In [8]:
from src.trigger_engine import compute_p_risk_snapshot

snap = compute_p_risk_snapshot(df_hourly, ultima_troca, hoje)
print("Indicadores calculados:")
for k, v in snap.items():
    print(f"  {k:<22}: {v}")

Indicadores calculados:
  today                 : 2026-05-25
  age_days              : 6
  mean_3d               : 1180.7
  mean_7d               : 1165.1
  mean_14d              : 1249.6
  min_3d                : 731.9
  mediana_3d            : 1272.0
  n_leituras_abaixo_800 : 1
  ratio_3_14            : 0.9448
  slope_7d              : 12.9
  proj_48h              : 1206.5
  age_risk              : 0.0719
  sig_score             : 0.0331
  p_risk                : 0.0918
  cond_p_risk           : False
  cond_signal           : False
  cond_idade            : False
  cond_proj             : False
  cond_critico          : True
  cond_risco            : True


In [ ]:
# ── Histórico de gatilhos por ciclo ────────────────────────────────────────
#   0 = ciclo atual (aberto)
#  -1 = último ciclo completo
#  -2 = penúltimo  ...
CICLO_OFFSET = -1

import tempfile, json
from pathlib import Path
from src.trigger_engine import TriggerEngine, compute_p_risk_snapshot

# ── Resolver limites do ciclo ─────────────────────────────────────────────
_n = len(troca_dates)
_min_offset = -(_n - 1)   # ex: 15 trocas → offset mínimo = -14

if CICLO_OFFSET > 0 or CICLO_OFFSET < _min_offset:
    raise ValueError(
        f"CICLO_OFFSET deve estar entre {_min_offset} e 0 "
        f"(há {_n - 1} ciclos completos + ciclo atual)."
    )

if CICLO_OFFSET == 0:
    t_ini  = troca_dates[-1]
    t_fim  = None
    rotulo = f"atual (desde {t_ini.date()})"
else:
    t_ini  = troca_dates[CICLO_OFFSET - 1]
    t_fim  = troca_dates[CICLO_OFFSET]
    _dur   = (t_fim - t_ini).days
    rotulo = f"ciclo {CICLO_OFFSET:+d}  |  {t_ini.date()} → {t_fim.date()}  ({_dur} dias)"

print(f"━━ Ciclo selecionado: {rotulo} ━━\n")

# ── Filtrar séries do ciclo ───────────────────────────────────────────────
_df_c = df_hourly[df_hourly.index >= t_ini]
if t_fim:
    _df_c = _df_c[_df_c.index < t_fim]

if _df_c.empty:
    print("  Sem dados para este ciclo.")
else:
    # Mini-backtest: reconstruir eventos disparados no ciclo
    _state_tmp = Path(tempfile.mktemp(suffix=".json"))
    _eng       = TriggerEngine(MAQUINA, _state_tmp)
    _dias      = pd.DatetimeIndex(sorted({ts.normalize() for ts in _df_c.index}))

    _eventos = []   # lista de (day, TriggerEvent)
    for _day in _dias:
        _df_ate = df_hourly[df_hourly.index <= _day + pd.Timedelta(hours=23, minutes=59)]
        try:
            for _ev in _eng.evaluate(
                _df_ate,
                troca_date=t_ini,
                sp_client=None,
                today=_day,
                troca_dates=troca_dates,
                media_col=media_col,
            ):
                _eventos.append((_day, _ev))
        except Exception as _exc:
            print(f"  [ERRO] {_day.date()}: {_exc}")

    if _state_tmp.exists():
        _state_tmp.unlink()

    # ── Sumário ─────────────────────────────────────────────────────────
    if not _eventos:
        print("  Nenhum gatilho disparado neste ciclo.")
    else:
        print(f"  {len(_eventos)} disparo(s):\n")
        for _d, _ev in _eventos:
            _mark = "  ◀ último" if (_d, _ev) is _eventos[-1] else ""
            print(f"    {_d.date()}  [{_ev.severidade:<6}]  {_ev.gatilho}{_mark}")

        # ── Detalhe do último disparo ─────────────────────────────────
        _day_ult, _ev_ult = _eventos[-1]
        print(f"\n── Último disparo: {_ev_ult.gatilho}  "
              f"em {_day_ult.date()}  (dia {_ev_ult.idade_maintacker} do ciclo) ──\n")

        _snap = compute_p_risk_snapshot(df_hourly, t_ini, _day_ult)
        for _k, _v in _snap.items():
            print(f"  {_k:<22}: {_v}")

        print("\n── Payload enviado (TeamsPayload) ──")
        if _ev_ult.teams_payload:
            try:
                print(json.dumps(json.loads(_ev_ult.teams_payload), indent=2, ensure_ascii=False))
            except Exception:
                print(_ev_ult.teams_payload)
        else:
            print("  (sem payload — dry-run)")